In [4]:
import sys
sys.path.append(r'C:\Users\fator\Desktop\Proyectos1\Estructura_Datos\2026_02')

from goodrich.ch06.array_stack import ArrayStack
from goodrich.ch06.array_queue import ArrayQueue
from goodrich.exceptions import Empty


class InvalidRecordError(ValueError):
    """Error para registros que no tienen tres componentes o cuyo
    value no es numerico."""
    pass


class DataProcessor:

    def __init__(self):
        self._pendientes_Q = ArrayQueue()
        self._historial_P = ArrayStack()
        self._rehacer_P = ArrayStack()
        self._estado = []

    def _buscar_indice(self, sensor, variable):
        n = len(self._estado)
        i = 0
        while i < n:
            actual = self._estado[i]
            if actual[0] == sensor and actual[1] == variable:
                return i
            i += 1
        return None

    def _vaciar_pila(self, pila):
        while len(pila) > 0:
            pila.pop()

    def add(self, record):
        if len(record) != 3:
            raise InvalidRecordError(
                "El registro debe tener exactamente tres componentes."
            )

        if isinstance(record[2], bool) or not isinstance(record[2], (int, float)):
            raise InvalidRecordError("El campo value debe ser numerico.")

        self._pendientes_Q.enqueue(record)

    def process_next(self):
        record = self._pendientes_Q.dequeue()
        sensor, variable, value = record

        indice = self._buscar_indice(sensor, variable)

        if indice is None:
            self._estado.append(record)
            nuevo_indice = len(self._estado) - 1
            self._historial_P.push(("crear", nuevo_indice, record))
        else:
            valor_anterior = self._estado[indice]
            self._estado[indice] = record
            self._historial_P.push(("actualizar", indice, valor_anterior, record))

        self._vaciar_pila(self._rehacer_P)

        return record

    def undo(self):
        accion = self._historial_P.pop()
        tipo = accion[0]

        if tipo == "crear":
            indice = accion[1]
            self._estado.pop(indice)
        else:
            indice = accion[1]
            valor_anterior = accion[2]
            self._estado[indice] = valor_anterior

        self._rehacer_P.push(accion)

    def redo(self):
        accion = self._rehacer_P.pop()
        tipo = accion[0]

        if tipo == "crear":
            indice, record = accion[1], accion[2]
            self._estado.insert(indice, record)
        else:
            indice, valor_nuevo = accion[1], accion[3]
            self._estado[indice] = valor_nuevo

        self._historial_P.push(accion)

    def pending(self):
        return len(self._pendientes_Q)

    def current_value(self, sensor, variable):
        indice = self._buscar_indice(sensor, variable)
        if indice is None:
            raise KeyError((sensor, variable))
        return self._estado[indice][2]


In [5]:
A = ("S01", "temperature", 20)
B = ("S01", "temperature", 25)
C = ("S01", "humidity", 60)

# 1. pending() sobre un procesador vacio
p = DataProcessor()
assert p.pending() == 0

# 2. agregar un registro
p = DataProcessor()
p.add(A)
assert p.pending() == 1

# 3. agregar varios registros
p = DataProcessor()
p.add(A)
p.add(B)
p.add(C)
assert p.pending() == 3

# 4. procesamiento FIFO
p = DataProcessor()
p.add(A)
p.add(B)
p.add(C)
assert p.process_next() == A
assert p.process_next() == B
assert p.process_next() == C

# 5. procesar un registro
p = DataProcessor()
p.add(A)
p.process_next()
assert p.current_value("S01", "temperature") == 20

# 6. procesar varios registros
p = DataProcessor()
p.add(A)
p.add(C)
p.process_next()
p.process_next()
assert p.current_value("S01", "temperature") == 20
assert p.current_value("S01", "humidity") == 60

# 7. actualizar una variable existente
p = DataProcessor()
p.add(A)
p.add(B)
p.process_next()
p.process_next()
assert p.current_value("S01", "temperature") == 25

# 8. consultar el valor actual
p = DataProcessor()
p.add(C)
p.process_next()
assert p.current_value("S01", "humidity") == 60

# 9. undo()
p = DataProcessor()
p.add(A)
p.process_next()
p.undo()
try:
    p.current_value("S01", "temperature")
    assert False
except KeyError:
    pass

# 10. varios undo() consecutivos
p = DataProcessor()
p.add(A)
p.add(B)
p.add(C)
p.process_next()
p.process_next()
p.process_next()
p.undo()
p.undo()
p.undo()
try:
    p.current_value("S01", "temperature")
    assert False
except KeyError:
    pass
try:
    p.current_value("S01", "humidity")
    assert False
except KeyError:
    pass

# 11. process_next() con Queue vacia
p = DataProcessor()
try:
    p.process_next()
    assert False
except Empty:
    pass

# 12. undo() con historial vacio
p = DataProcessor()
try:
    p.undo()
    assert False
except Empty:
    pass

# 13. deshacer la creacion de un dato nuevo
p = DataProcessor()
p.add(C)
p.process_next()
p.undo()
try:
    p.current_value("S01", "humidity")
    assert False
except KeyError:
    pass

# 14. varios cambios sobre la misma variable
p = DataProcessor()
p.add(A)
p.add(B)
p.add(("S01", "temperature", 30))
p.process_next()
p.process_next()
p.process_next()
assert p.current_value("S01", "temperature") == 30
p.undo()
assert p.current_value("S01", "temperature") == 25
p.undo()
assert p.current_value("S01", "temperature") == 20

# 15. registro con formato incorrecto
p = DataProcessor()
try:
    p.add(("S01", "temperature"))
    assert False
except InvalidRecordError:
    pass

# 16. registro con value no numerico
p = DataProcessor()
try:
    p.add(("S01", "temperature", "alto"))
    assert False
except InvalidRecordError:
    pass

# 17. consultar sensor/variable nunca procesado
p = DataProcessor()
try:
    p.current_value("S99", "presion")
    assert False
except KeyError:
    pass

# bonus: redo()
p = DataProcessor()
p.add(A)
p.process_next()
p.undo()
p.redo()
assert p.current_value("S01", "temperature") == 20

print("Todas las pruebas pasaron correctamente.")


Todas las pruebas pasaron correctamente.


## 3. Decisiones de diseño

- El estado actual se guarda en una lista de tuplas `(sensor, variable, value)`, tal como pide el enunciado. Para saber si un `(sensor, variable)` ya existe, se recorre la lista de manera secuencial.
- El historial guarda, para cada cambio, si fue una **creación** o una **actualización**, junto con la posición en la lista y el valor anterior (cuando aplica). Esto permite que `undo()` sepa exactamente qué deshacer: si fue creación, se elimina el dato; si fue actualización, se restaura el valor anterior.
- Como los datos solo se agregan al final de la lista (`append`) y `undo()` deshace siempre el cambio más reciente (orden LIFO), la posición que hay que eliminar al deshacer una creación siempre corresponde al último elemento de la lista en ese momento, por lo que no hay que reacomodar índices.
- Para el bonus `redo()` se usa una segunda `ArrayStack`. Cada vez que se deshace un cambio con `undo()`, ese cambio se guarda ahí para poder reaplicarlo. Si se procesa un registro nuevo, esa pila se vacía, porque ya no tendría sentido rehacer algo de una rama de cambios que quedó atrás.

## 4. Análisis de complejidad

| Método | Complejidad | Justificación |
|---|---|---|
| `add` | O(1) | Solo valida el tamaño del registro y el tipo de `value`, y encola (operación O(1) de la `ArrayQueue`). |
| `process_next` | O(n) | La búsqueda de `(sensor, variable)` en la lista de estado recorre hasta n elementos en el peor caso. |
| `undo` / `redo` | O(1) amortizado | `push`/`pop` en la `ArrayStack` son O(1); la posición modificada en la lista de estado siempre es la última, por el orden LIFO. |
| `pending` | O(1) | Es directamente `len` de la `ArrayQueue`. |
| `current_value` | O(n) | Usa la misma búsqueda lineal que `process_next`. |

La operación que domina el costo es la búsqueda lineal sobre la lista de estado, ya que el enunciado exige usar una lista (no un diccionario) para guardar los datos actuales.
